# 08 — Credential Dump Parser

Credential exposure is the single highest-value dark-web signal for a SOC. If an employee's corporate email shows up in a stealer log, the org is one unpatched reused-password away from a breach.

This notebook builds the parsing side of that pipeline:

1. **Generate synthetic dumps** in the four common formats: combo lists, stealer logs, columnar CSV dumps, and RedLine-style archives. Real leaked credentials describe real victims — we never use them in teaching material, and the *parsing* problem is about format anyway.
2. **Parser** — one normalized `CredentialRecord` schema across all four formats, with URL-to-domain canonicalization and email validation.
3. **Customer match** — reuse the synthetic customer profile from nb 06 and flag any records hitting employee emails or corporate domains.
4. **HIBP breach context** — fetch the public breach list from Have I Been Pwned (no auth needed) and annotate matches with *"this type of breach typically exposes X"* — adding real-world context without exposing real victims.

## Why synthetic

Real stealer logs and combo lists contain genuinely compromised people. Using them in a teaching notebook — even "publicly available" ones — is harmful. Synthetic data teaches the *same* parsing lesson, with zero victimization risk. The code transfers directly to real data.

## 1 — Synthetic dump generator

In [ ]:
import json, re, random, string, os
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Optional
from urllib.parse import urlparse
import pandas as pd

random.seed(1337)

SAMPLE_DIR = Path('data/synthetic_dumps'); SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# --- common services + domains to populate realistic-looking dumps ---
SERVICES = [
    ('gmail.com', 'https://accounts.google.com/signin'),
    ('yahoo.com', 'https://login.yahoo.com'),
    ('outlook.com', 'https://login.live.com'),
    ('netflix.com', 'https://www.netflix.com/login'),
    ('spotify.com', 'https://accounts.spotify.com/login'),
    ('amazon.com', 'https://www.amazon.com/signin'),
    ('dropbox.com', 'https://www.dropbox.com/login'),
    ('facebook.com', 'https://www.facebook.com/login'),
    ('linkedin.com', 'https://www.linkedin.com/uas/login'),
    ('acme-industrial.com', 'https://mail.acme-industrial.com/owa'),   # customer hit
    ('acmeind.com', 'https://vpn.acmeind.com'),                        # customer hit
]
USERNAMES = ['jsmith', 'olivia99', 'darkowl', 'mike.anderson', 'lisa_martinez',
             'crypto.king', 'nancy.clark', 'pilotpete', 'sarah.j', 'user7743']
PASSWORDS = ['Password1', 'Sunshine!', 'Letmein2024', 'qwerty123', 'Dragon99',
             'Winter2025!', 'P@ssw0rd', 'abc123', 'MyDog2020', 'S3cret!']

def rand_email(domain=None):
    user = random.choice(USERNAMES) + (str(random.randint(1, 999)) if random.random() < 0.4 else '')
    d = domain or random.choice(SERVICES)[0]
    return f'{user}@{d}'

def rand_password():
    return random.choice(PASSWORDS) + (str(random.randint(10,9999)) if random.random() < 0.3 else '')

# --- Format 1: combo list (email:password) ---------------------------
def gen_combo_list(path, n=2000, customer_hit_rate=0.02):
    with open(path, 'w') as f:
        for _ in range(n):
            if random.random() < customer_hit_rate:
                dom = random.choice(['acme-industrial.com', 'acmeind.com'])
                f.write(f'{rand_email(dom)}:{rand_password()}\n')
            else:
                f.write(f'{rand_email()}:{rand_password()}\n')

# --- Format 2: stealer log (url;user;password) ------------------------
def gen_stealer_log(path, n=1500):
    with open(path, 'w') as f:
        for _ in range(n):
            dom, url = random.choice(SERVICES)
            # Some rows use the service domain as email, some use gmail with that service
            if random.random() < 0.5:
                email = rand_email(dom)
            else:
                email = rand_email()  # third-party email at the service
            f.write(f'{url};{email};{rand_password()}\n')

# --- Format 3: columnar CSV dump -------------------------------------
def gen_csv_dump(path, n=1000):
    rows = []
    for i in range(n):
        dom, url = random.choice(SERVICES)
        rows.append({
            'id': i, 'email': rand_email(dom), 'password_hash': rand_password(),
            'source_url': url, 'breach_date': random.choice(['2024-03-01','2024-09-15','2025-01-22']),
        })
    pd.DataFrame(rows).to_csv(path, index=False)

# --- Format 4: RedLine-style folder (one dir per infected host) ------
def gen_redline_archive(root, n_hosts=20, entries_per_host=(5, 30)):
    for h in range(n_hosts):
        host = f'DESKTOP-{"".join(random.choices(string.ascii_uppercase+string.digits, k=7))}'
        d = root / host
        d.mkdir(exist_ok=True)
        # UserInformation.txt (recon metadata)
        (d / 'UserInformation.txt').write_text(
            f'Machine: {host}\nOS: Windows 10 x64\nIP: {".".join(str(random.randint(1,255)) for _ in range(4))}\n'
            f'Country: {random.choice(["US","BR","IN","DE","GB"])}\n')
        # Passwords.txt (URL=..., USER=..., PASS=... blocks)
        n = random.randint(*entries_per_host)
        blocks = []
        for _ in range(n):
            dom, url = random.choice(SERVICES)
            blocks.append(f'URL: {url}\nUSER: {rand_email(dom)}\nPASS: {rand_password()}\n\n')
        (d / 'Passwords.txt').write_text(''.join(blocks))

# --- Generate all four ---
gen_combo_list(SAMPLE_DIR / 'combo_2025_04.txt')
gen_stealer_log(SAMPLE_DIR / 'redline_logs_pack.txt')
gen_csv_dump(SAMPLE_DIR / 'megadump.csv')
redline_root = SAMPLE_DIR / 'redline_archive'; redline_root.mkdir(exist_ok=True)
gen_redline_archive(redline_root)

print('Generated:')
for p in sorted(SAMPLE_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(SAMPLE_DIR)}  ({p.stat().st_size:,} bytes)')

# Peek at each
print('\n--- combo ---'); print((SAMPLE_DIR / 'combo_2025_04.txt').read_text().splitlines()[0])
print('--- stealer log ---'); print((SAMPLE_DIR / 'redline_logs_pack.txt').read_text().splitlines()[0])
print('--- csv (header + row) ---')
with open(SAMPLE_DIR / 'megadump.csv') as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i == 1: break
print('--- redline passwords.txt (first 3 lines) ---')
some_host = next(redline_root.iterdir())
print((some_host / 'Passwords.txt').read_text()[:200])

## 2 — The parser: one schema across four formats

In [ ]:
EMAIL_RE = re.compile(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')

@dataclass
class CredentialRecord:
    email:     Optional[str]
    username:  Optional[str]
    password:  Optional[str]
    url:       Optional[str]
    domain:    Optional[str]
    source:    str
    source_format: str    # combo / stealer / csv / redline
    host_id:   Optional[str] = None

def url_to_domain(url):
    if not url: return None
    try:
        return urlparse(url if '://' in url else 'http://' + url).netloc.lower() or None
    except Exception:
        return None

def email_to_domain(email):
    return email.split('@', 1)[1].lower() if email and '@' in email else None

def is_email(s):
    return bool(s and EMAIL_RE.match(s))

def parse_combo_list(path):
    src = str(path)
    with open(path) as f:
        for line_no, raw in enumerate(f, 1):
            line = raw.strip()
            if not line or ':' not in line: continue
            user, pw = line.split(':', 1)
            if not pw: continue
            email = user if is_email(user) else None
            yield CredentialRecord(
                email=email, username=None if email else user, password=pw,
                url=None, domain=email_to_domain(email),
                source=f'{src}:{line_no}', source_format='combo',
            )

def parse_stealer_log(path):
    src = str(path)
    with open(path) as f:
        for line_no, raw in enumerate(f, 1):
            parts = raw.strip().split(';')
            if len(parts) < 3: continue
            url, user, pw = parts[0], parts[1], ';'.join(parts[2:])
            email = user if is_email(user) else None
            yield CredentialRecord(
                email=email, username=None if email else user, password=pw,
                url=url, domain=url_to_domain(url) or email_to_domain(email),
                source=f'{src}:{line_no}', source_format='stealer',
            )

def parse_csv_dump(path):
    src = str(path)
    df = pd.read_csv(path)
    EMAIL_COLS = ['email', 'user', 'username', 'login']
    PW_COLS = ['password', 'password_hash', 'pass', 'pw']
    URL_COLS = ['source_url', 'url', 'origin', 'site']
    def first(row, cols):
        for c in cols:
            if c in row and pd.notna(row[c]):
                return row[c]
        return None
    for i, row in df.iterrows():
        user = first(row, EMAIL_COLS)
        email = user if is_email(user) else None
        pw = first(row, PW_COLS)
        url = first(row, URL_COLS)
        yield CredentialRecord(
            email=email, username=None if email else user,
            password=str(pw) if pw is not None else None,
            url=url, domain=url_to_domain(url) or email_to_domain(email),
            source=f'{src}:{i+2}', source_format='csv',
        )

RED_URL = re.compile(r'^URL:\s*(.+)', re.IGNORECASE)
RED_USER = re.compile(r'^USER:\s*(.+)', re.IGNORECASE)
RED_PASS = re.compile(r'^PASS:\s*(.+)', re.IGNORECASE)

def parse_redline_archive(root):
    for host_dir in Path(root).iterdir():
        if not host_dir.is_dir(): continue
        pw_file = host_dir / 'Passwords.txt'
        if not pw_file.exists(): continue
        lines = pw_file.read_text().splitlines()
        buf = {}
        for line in lines + ['']:
            if not line.strip():
                if buf.get('USER') and buf.get('PASS'):
                    user = buf['USER']
                    email = user if is_email(user) else None
                    yield CredentialRecord(
                        email=email, username=None if email else user, password=buf['PASS'],
                        url=buf.get('URL'),
                        domain=url_to_domain(buf.get('URL')) or email_to_domain(email),
                        source=str(pw_file), source_format='redline', host_id=host_dir.name,
                    )
                buf = {}
                continue
            for pat, key in [(RED_URL, 'URL'), (RED_USER, 'USER'), (RED_PASS, 'PASS')]:
                m = pat.match(line)
                if m:
                    buf[key] = m.group(1).strip()
                    break

# --- Run all parsers ---
records = []
records += list(parse_combo_list(SAMPLE_DIR / 'combo_2025_04.txt'))
records += list(parse_stealer_log(SAMPLE_DIR / 'redline_logs_pack.txt'))
records += list(parse_csv_dump(SAMPLE_DIR / 'megadump.csv'))
records += list(parse_redline_archive(redline_root))

print(f'Parsed {len(records)} credential records')
print('\nBy format:')
print(pd.Series([r.source_format for r in records]).value_counts().to_string())

## 3 — Dedupe + domain-level summary

In [ ]:
df_rec = pd.DataFrame([asdict(r) for r in records])

# Dedupe on (email|username, password, domain) — same cred appearing in 2 formats is one fact
before = len(df_rec)
df_rec['dedupe_key'] = (df_rec['email'].fillna(df_rec['username']).fillna('') + '|'
                         + df_rec['password'].fillna('') + '|'
                         + df_rec['domain'].fillna(''))
df_dedup = df_rec.drop_duplicates(subset=['dedupe_key']).drop(columns=['dedupe_key'])
print(f'Dedup: {before} -> {len(df_dedup)} records')

print('\nTop 15 exposed domains:')
print(df_dedup['domain'].value_counts().head(15).to_string())

## 4 — Match against the nb 06 customer profile

In [ ]:
# Reuse the same customer def as nb 06 (inline here so notebook is standalone)
CUSTOMER = {
    'name':     'Acme Industrial Corp',
    'domains':  ['acme-industrial.com', 'acmeind.com', 'acme-corp.net'],
    'known_employee_emails': [],  # in a real deploy this is the HR roster
}
cust_domains = set(CUSTOMER['domains'])

def is_customer_hit(rec):
    dom = (rec['domain'] or '').lower()
    if dom in cust_domains:
        return 'employee_credential'
    email = (rec['email'] or '').lower()
    if email and any(email.endswith('@' + d) for d in cust_domains):
        return 'employee_credential'
    url_dom = (rec['url'] or '').lower()
    for d in cust_domains:
        if d in url_dom:
            return 'customer_url_login'   # someone logged in to a customer-facing site
    return None

df_dedup['hit_type'] = df_dedup.apply(is_customer_hit, axis=1)
hits = df_dedup[df_dedup['hit_type'].notna()].copy()

print(f'Customer hits: {len(hits)} records across {hits["email"].nunique()} distinct emails')
print('\nHit types:')
print(hits['hit_type'].value_counts().to_string())

print('\nSample hits:')
for _, h in hits.head(8).iterrows():
    pw_masked = (h['password'][:2] + '***' + h['password'][-1]) if h['password'] else None
    print(f'  [{h["hit_type"]:<22s}] {h["email"] or h["username"]}  pw={pw_masked}  src={h["source_format"]}')

## 5 — HIBP breach context

For each domain with hits, check if it appears in the public Have I Been Pwned breach list — and enrich alerts with *what categories of data that breach historically exposed*. This is the real-world framing without real-world victims.

In [ ]:
import urllib.request

HIBP_CACHE = Path('processed/hibp_breaches.json')
HIBP_CACHE.parent.mkdir(exist_ok=True)

if not HIBP_CACHE.exists():
    print('Fetching HIBP public breach list (one-time)...')
    req = urllib.request.Request('https://haveibeenpwned.com/api/v3/breaches',
                                  headers={'User-Agent': 'cti301-educational'})
    data = urllib.request.urlopen(req, timeout=30).read()
    HIBP_CACHE.write_bytes(data)

hibp = json.loads(HIBP_CACHE.read_text())
print(f'{len(hibp)} breaches in HIBP catalog')

# Build domain -> breach record lookup
hibp_by_domain = defaultdict(list)
for b in hibp:
    if b.get('Domain'):
        hibp_by_domain[b['Domain'].lower()].append(b)

# Which of our exposed domains are in HIBP?
exposed_domains = df_dedup['domain'].dropna().unique()
in_hibp = [(d, hibp_by_domain[d][0]) for d in exposed_domains if d in hibp_by_domain]
print(f'\n{len(in_hibp)} of our {len(exposed_domains)} exposed domains appear in HIBP catalog:')
for d, b in in_hibp[:15]:
    print(f'  {d:<30s} breached {b["BreachDate"]}  ({b["PwnCount"]:>12,} accounts)  data={",".join(b["DataClasses"][:4])}')

# Enrich customer hits where the victim domain has a public breach context
def enrich_with_hibp(rec):
    dom = (rec['domain'] or '').lower()
    if dom in hibp_by_domain:
        b = hibp_by_domain[dom][0]
        return {
            'hibp_breach': b['Name'],
            'hibp_date':   b['BreachDate'],
            'hibp_pwn_count': b['PwnCount'],
            'hibp_data_classes': b['DataClasses'],
        }
    return None

## 6 — Emit SOC-ready credential alerts

In [ ]:
import hashlib

def severity(hit):
    if hit['hit_type'] == 'employee_credential':
        return 'critical'
    if hit['hit_type'] == 'customer_url_login':
        return 'high'
    return 'medium'

cred_alerts = []
for _, h in hits.iterrows():
    enrich = enrich_with_hibp(h)
    alert = {
        'alert_id': hashlib.sha256(f'{h["email"]}{h["password"]}{h["source"]}'.encode()).hexdigest()[:12],
        'source':   'credential-dump-pipeline',
        'customer': CUSTOMER['name'],
        'severity': severity(h),
        'hit_type': h['hit_type'],
        'exposed_email': h['email'],
        'exposed_username': h['username'] if not h['email'] else None,
        'exposed_domain': h['domain'],
        'password_preview': (h['password'][:2] + '***' + h['password'][-1]) if h['password'] else None,
        'source_format': h['source_format'],
        'source_file': h['source'],
    }
    if enrich:
        alert['hibp_context'] = enrich
    cred_alerts.append(alert)

OUT = Path('processed/credential_alerts.jsonl')
with open(OUT, 'w') as f:
    for a in cred_alerts:
        f.write(json.dumps(a) + '\n')

print(f'Wrote {len(cred_alerts)} credential alerts to {OUT}')
print('\nSeverity distribution:')
print(pd.Series([a["severity"] for a in cred_alerts]).value_counts().to_string())

print('\nSample alert (with HIBP context if available):')
enriched = [a for a in cred_alerts if 'hibp_context' in a]
print(json.dumps(enriched[0] if enriched else cred_alerts[0], indent=2))

## What you've done

- Generated **four common credential-dump formats** synthetically — combo lists, stealer logs, columnar CSV, and RedLine-style host archives — without touching real leaked data.
- Wrote format-specific parsers that emit a **uniform `CredentialRecord` schema**, with URL→domain canonicalization and email validation.
- Deduped and summarized by domain.
- Matched against the **nb 06 customer profile** to surface employee-credential exposures.
- Enriched alerts with **HIBP breach context** from the public API.
- Emitted **SOC-ready JSON alerts** consistent with the format from nb 06.

## Honest caveats

- **Synthetic means easier.** Real dumps have more noise: binary junk, mixed encodings, obfuscated delimiters, partial rows. The parser would need robustness layers (try/except per line, encoding detection) to survive in production.
- **No credential validation.** We don't check whether the leaked password is current, reused elsewhere, or actually works. Real pipelines integrate with AD / password-hash lookups to score severity.
- **Password masking is deliberate.** Real alerts to a SOC analyst should NEVER contain plaintext passwords — the masking pattern here is the bare minimum.

## What's next

- **Nb 09 — End-to-end SOC pipeline.** Chain nb 03 (classification) → nb 05 (extraction) → nb 06 (asset matching) → nb 07 (leak-site detection) → nb 08 (credentials) into a single orchestrator that ingests raw dark-web text and emits a ranked alert feed.